# 09 — Neural network calibration (stretch)

Horvath-Muguruza-Tomas (2019) demonstrated that a small MLP can invert the price -> param map for rough vol models in milliseconds, ~1000x faster than traditional optimization. Useful for portfolio sensitivity work where calibration must run thousands of times.

Requires torch — install via `pip install volengine[neural]`.

## Context (optional stretch)

Following Horvath–Muguruza–Tomas (2019), we train a small fully-connected MLP that maps an IV grid directly to rBergomi parameters. Inference is then a few milliseconds — order $10^3 \times$ faster than the traditional DE → L-BFGS-B pipeline.

This notebook is intentionally illustrative: training data is generated from a uniform parameter grid (production would use stratified sampling), and the architecture is a 3-layer MLP (production would use deeper / residual). The goal is to demonstrate the technique, not to ship a deployable calibrator.

Requires `pip install volengine[neural]` (PyTorch).

In [ ]:
import numpy as np
import time

from volengine.neural.data_generation import TrainingGrid
from volengine.neural import train_calibrator

In [ ]:
grid = TrainingGrid(
    maturities=np.array([0.1, 0.25, 0.5, 1.0]),
    log_moneyness=np.linspace(-0.2, 0.2, 9),
)
t0 = time.time()
calibrator = train_calibrator(grid, n_samples=500, epochs=100)  # small for demo
print(f'Trained in {time.time() - t0:.1f}s')

## Loss curve

**Figure.** Training MSE versus epoch on normalized parameters. We expect rapid convergence — the inverse map is smooth and 4-dimensional.

In [ ]:
from volengine.models.rbergomi import RBergomiParameters, simulate_rbergomi
from volengine.surfaces import implied_vol

true_p = RBergomiParameters(H=0.12, eta=2.1, rho=-0.85, xi0=0.045)
iv_grid = np.zeros((len(grid.maturities), len(grid.log_moneyness)))
S0, r, q = 100.0, 0.0, 0.0
for i, T in enumerate(grid.maturities):
    F = S0 * np.exp((r - q) * T)
    n_steps = max(20, int(80 * T))
    S = simulate_rbergomi(S0, T, true_p, r, q, 10_000, n_steps, seed=0)
    ST = S[:, -1]
    for j, k in enumerate(grid.log_moneyness):
        K = F * np.exp(k)
        P = np.exp(-r * T) * np.maximum(ST - K, 0).mean()
        iv_grid[i, j] = implied_vol(float(P), S0, K, T, r, q, 'call')

t1 = time.time()
inferred = calibrator.calibrate(iv_grid)
print(f'NN inversion: {(time.time() - t1) * 1000:.2f} ms')
print(f'True:    {true_p}')
print(f'NN-fit:  {inferred}')

## NN vs. classical calibration

Speed comparison on a held-out market surface: classical (DE → L-BFGS-B, $\sim 30$ s) vs. NN inversion ($\sim 10$ ms). The classical result is the *ground truth*; the NN result should match within $\sim 5\%$ relative error on each parameter for a well-trained network.